# CAISO Monthly Data — EDA

Replicating *Integrating Kolmogorov–Arnold Networks with Time Series Prediction Framework in Electricity Demand Forecasting* (Zhang et al., 2025) with CAISO data.

**Run `python scripts/fetch_caiso.py` first** to populate `data/raw/caiso_monthly.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_regression

sns.set_theme(style='whitegrid', palette='tab10')
%matplotlib inline

## 1. Load CAISO data

In [ ]:
caiso = pd.read_csv('../data/raw/caiso_monthly.csv', parse_dates=['date'])
caiso = caiso.set_index('date').sort_index()
print(caiso.shape)
caiso.head()

In [ ]:
caiso.describe().round(2)

## 2. Missing value summary (mirrors Table 2 in paper)

In [ ]:
missing = caiso.isnull().mean() * 100
print(missing.round(1).to_string())

## 3. Time-series plots (mirrors Figure 1 in paper)

In [ ]:
feature_cols = [c for c in caiso.columns if c not in ('year', 'month')]
target = 'load_mw'

fig, axes = plt.subplots(len(feature_cols), 1, figsize=(12, 3 * len(feature_cols)), sharex=True)
if len(feature_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, feature_cols):
    ax.plot(caiso.index, caiso[col], label=col, color='steelblue')
    if col != target:
        ax2 = ax.twinx()
        ax2.plot(caiso.index, caiso[target], label=target, color='tomato', alpha=0.5, linewidth=0.8)
    ax.set_ylabel(col, fontsize=9)
    ax.set_xlabel('')

plt.tight_layout()
plt.savefig('../data/processed/time_series_overview.png', dpi=150)
plt.show()

## 4. Correlation heat map (mirrors Figure 2 in paper)

In [ ]:
numeric = caiso[feature_cols].dropna()
corr = numeric.corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Pearson Correlation — CAISO Monthly Features')
plt.tight_layout()
plt.savefig('../data/processed/pearson_heatmap.png', dpi=150)
plt.show()

print('\nCorrelation with load_mw:')
print(corr[target].drop(target).sort_values(ascending=False).round(3))

## 5. Mutual information (mirrors Figure 3 in paper)

In [ ]:
feats = [c for c in feature_cols if c != target]
X_mi = numeric[feats].fillna(numeric[feats].median())
y_mi = numeric[target].fillna(numeric[target].median())

mi_scores = mutual_info_regression(X_mi, y_mi, random_state=42)
mi_df = pd.Series(mi_scores, index=feats).sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
mi_df.plot.barh(ax=ax, color='teal')
ax.set_xlabel('Mutual Information')
ax.set_title('Mutual Information with load_mw')
plt.tight_layout()
plt.savefig('../data/processed/mutual_information.png', dpi=150)
plt.show()
print(mi_df.round(3))

## 6. Seasonality check

In [ ]:
caiso['month_num'] = caiso.index.month
monthly_avg = caiso.groupby('month_num')[target].mean()

fig, ax = plt.subplots(figsize=(8, 4))
monthly_avg.plot(ax=ax, marker='o', color='steelblue')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_ylabel('Avg Load (MW)')
ax.set_title('CAISO Average Monthly Load (2015–2024)')
plt.tight_layout()
plt.savefig('../data/processed/seasonality.png', dpi=150)
plt.show()

## 7. Save processed feature table

Rename columns to match the paper's convention and export for use in modeling.

In [ ]:
rename_map = {
    'load_mw':        'nd',      # national/total demand (target)
    'renewables_mw':  'x3',      # NP-analog: renewable generation
    'gas_mw':         'x4',      # SP-analog: gas generation
    'net_imports_mw': 'x9',      # transmission system demand
}

out = caiso[list(rename_map)].rename(columns=rename_map).dropna(subset=['nd'])
print(out.shape)
out.head()

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)
out.to_csv('../data/processed/caiso_features.csv')
print('Saved → data/processed/caiso_features.csv')